# Tutorial: IDF curves for Québec City, end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rare-labs/tails/blob/main/docs/notebooks/tutorial_quebec_idf.ipynb)

A worked answer to *how do I build an IDF curve for a real Canadian station?* using the bundled ECCC record at Québec City (Jean Lesage, `701S001`).

The same walkthrough, with the figures already drawn, is on the [docs site](https://rare-labs.github.io/tails/tutorial_quebec_idf/). This notebook recomputes it.

We estimate rainfall intensity–duration–frequency (IDF) curves for **Québec City – Jean Lesage International Airport (ECCC station 701S001, 1961–2024)**, compare models and their uncertainty, check for a trend, and then fit **all 151 Québec stations at once**.


## Setup

The import is `tails`. The distribution name is `rare-tails`. This cell installs the plotting extra from GitHub, which is where the package lives until the PyPI release is published (`pip install "rare-tails[plots]"` after that). The first fit compiles JAX and is slower than the later ones.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "rare-tails[plots] @ git+https://github.com/rare-labs/tails"]
)
print("Installed rare-tails[plots].")


In [ ]:
import time

try:
    from IPython import get_ipython

    _ipython = get_ipython()
    if _ipython is not None:
        _ipython.run_line_magic("matplotlib", "inline")
except Exception:
    pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2

import tails
from tails.datasets import DURATION_TAGS, DURATIONS_H, load_eccc_quebec
from tails.idf import IDFData, fit_idf, scaling_test
from tails.plotting import apply_axes_style, color

# ECCC Table 2a, 100-year Gumbel amounts (mm) for 701S001, IDF v3.40 (2025-12-05).
ECCC_100 = {
    "5min": 17.3,
    "10min": 23.8,
    "15min": 28.9,
    "30min": 39.0,
    "1h": 50.5,
    "2h": 60.1,
    "6h": 75.4,
    "12h": 89.3,
    "24h": 104.1,
}


## 1. Data

ECCC's Engineering Climate Datasets (IDF v3.40, 2025-12-05) give, for each station, the annual maximum rainfall *depth* for 9 durations. `tails` bundles the Québec files and converts depths to intensities (mm/h).


In [ ]:
data = IDFData.from_eccc("701S001")
df = data.to_dataframe()
print(
    f"{data.n_years} years ({int(data.years.min())}–{int(data.years.max())}), "
    f"{int(np.isnan(data.values).sum())} missing values. First rows (mm/h):"
)
display(df.head(5).round(1))


## 2. One GEV per duration — nine fits in one call

Profile-likelihood intervals are asymmetric (longer upper tails), as they should be for 100-year levels. Wald intervals are symmetric by construction. ECCC's published values use a Gumbel (ξ = 0) fit by moments. The amounts below are intensities (mm/h).


In [ ]:
fit9 = tails.fit_gev(data.values.T)  # shape (9, 64): 9 series
rl = fit9.return_level([100], ci="profile")
rl_delta = fit9.return_level([100], ci="delta")
gev_table = pd.DataFrame(
    {
        "duration": data.tags,
        "shape ξ": fit9.params["shape"],
        "100-yr (mm/h)": rl.estimate[:, 0],
        "profile 95% CI": [f"[{a:.1f}, {b:.1f}]" for a, b in zip(rl.lower[:, 0], rl.upper[:, 0], strict=True)],
        "Wald 95% CI": [f"[{a:.1f}, {b:.1f}]" for a, b in zip(rl_delta.lower[:, 0], rl_delta.upper[:, 0], strict=True)],
        "ECCC Gumbel": [ECCC_100[tag] / hours for tag, hours in zip(DURATION_TAGS, DURATIONS_H, strict=True)],
    }
)
show = gev_table.copy()
for col, fmt in (("shape ξ", "{:.2f}"), ("100-yr (mm/h)", "{:.2f}"), ("ECCC Gumbel", "{:.2f}")):
    show[col] = show[col].map(lambda v, fmt=fmt: fmt.format(v))
display(show)


## 3. Which scaling model? (Scaling-model test)

The Zolotarev approximation is only meaningful for small p-values. Simple scaling is rejected at Québec City; the d-GEV is the model used below.


In [ ]:
p_simple, p_general = scaling_test(data, "simple"), scaling_test(data, "general")
print(f"p-values: simple scaling {p_simple:.3g}, general scaling (d-GEV) {p_general:.3g}")


## 4. d-GEV with and without cross-duration dependence

The copula log-likelihoods are far higher: maxima of different durations in the same year are strongly dependent (the same storm), which the independence model ignores. AIC of the independence model is not comparable — it is a composite likelihood.


In [ ]:
ind = fit_idf(data, "general")  # independence (composite likelihood)
gau = fit_idf(data, "general", "matern")  # Gaussian copula, Matérn on log-durations
stu = fit_idf(data, "general", "exponential", "t", df=5)
rows = []
for name, fit in (
    ("d-GEV, independent durations", ind),
    ("d-GEV + Matérn Gaussian copula", gau),
    ("d-GEV + exponential t(5) copula", stu),
):
    k = len(fit.names)
    rows.append(
        [name, *[f"{v:.4g}" for v in fit.params[0][:5]], f"{fit.loglik[0]:.2f}", f"{2 * k - 2 * fit.loglik[0]:.2f}"]
    )
display(pd.DataFrame(rows, columns=["model", "μ₀", "σ₀", "ξ", "α", "δ", "log-lik", "AIC"]))


## 5. How uncertain is the 100-year intensity? Three answers, one arbiter

The d-GEV fitted with independent durations needs a **sandwich (Godambe)** covariance; the naive Hessian pretends we have 9 × 64 independent observations. The copula model's Hessian is a proper likelihood Hessian. Because these Wald intervals can disagree by about 2×, the check is a **parametric bootstrap from the joint (copula) model**: simulate 64 years of dependent 9-duration maxima, refit, and repeat 500 times, all refits in one batched call.

The pooled models give a lower 1-hour 100-year intensity than both the single-duration GEV and ECCC's Gumbel value. The scaling relation borrows strength from other durations, and at Québec City the 1979 1-hour storm sits above the pooled curve. Pooling reduces variance but can bias individual durations.


In [ ]:
d = data.durations
t0 = time.perf_counter()
boot = gau.quantile_ci(d, [100], method="bootstrap", n_boot=500)
t_boot = time.perf_counter() - t0
cis = {
    "d-GEV naive Wald": ind.quantile_ci(d, [100], method="hessian"),
    "d-GEV sandwich": ind.quantile_ci(d, [100], method="sandwich"),
    "copula Wald": gau.quantile_ci(d, [100]),
    "copula bootstrap": boot,
}
rows = []
for i, tag in enumerate(data.tags):
    row = [
        tag,
        ECCC_100[tag] / DURATIONS_H[i],
        cis["d-GEV naive Wald"][0][0, i, 0],
        cis["copula Wald"][0][0, i, 0],
    ]
    for _name, (_est, lo, hi) in cis.items():
        row.append(f"[{lo[0, i, 0]:.1f}, {hi[0, i, 0]:.1f}]")
    rows.append(row)
display(
    pd.DataFrame(
        rows,
        columns=["duration", "ECCC Gumbel", "d-GEV estimate", "copula estimate", *cis.keys()],
    )
)
widths = ", ".join(
    f"{name} {hi[0, 4, 0] - lo[0, 4, 0]:.1f}" for name, (_est, lo, hi) in cis.items()
)
print(f"The 500 bootstrap refits took {t_boot:.1f} s. Interval widths at 1 h: {widths} mm/h.")


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.4))
x = np.arange(len(data.tags))
for j, (name, (_est, lo, hi)) in enumerate(cis.items()):
    y = cis["copula Wald"][0][0, :, 0] if "copula" in name else ind.quantile(d, [100])[0, :, 0]
    ax.errorbar(
        x + (j - 1.5) * 0.16,
        y,
        yerr=[y - lo[0, :, 0], hi[0, :, 0] - y],
        fmt="o",
        ms=4,
        capsize=2.5,
        elinewidth=1.0,
        color=color(j),
        label=name,
        zorder=3,
    )
ax.scatter(
    x,
    [ECCC_100[tag] / hours for tag, hours in zip(DURATION_TAGS, DURATIONS_H, strict=True)],
    marker="x",
    s=36,
    color="0.15",
    label="ECCC Gumbel",
    zorder=5,
)
ax.set_xticks(x, data.tags)
ax.set_yscale("log")
ax.set_ylabel("100-year intensity (mm/h)")
ax.set_xlabel("Duration")
ax.set_title("Québec City (701S001)\n100-year intensity and 95% intervals")
apply_axes_style(ax)
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.tight_layout()
plt.show()

ax = gau.plot(
    periods=(2, 10, 25, 100),
    ci="hessian",
    title="Québec City (701S001)\nIDF curves, d-GEV with Matérn Gaussian copula",
)
plt.show()


## 6. Is there a trend? A non-stationary GEV for every duration at once

A positive slope with a small p-value suggests the location of annual maxima is drifting up. With nine tests, apply a multiplicity correction before claiming one. The covariate is the year minus 2000, so the slope reported below is per decade.


In [ ]:
years = data.years - 2000.0
st = tails.fit_gev(data.values.T)
ns = tails.fit_gev(data.values.T, loc=years)  # mu(t) = b0 + b1 (t − 2000)
lr = 2 * (ns.loglik - st.loglik)  # likelihood-ratio statistic, 1 df
display(
    pd.DataFrame(
        {
            "duration": data.tags,
            "trend in μ (mm/h per decade)": 10 * ns.coef[:, 1],
            "s.e.": 10 * ns.se[:, 1],
            "LR p-value": chi2.sf(lr, 1),
        }
    )
)


## 7. Scale up: d-GEV for all 151 Québec stations in one call

Stations with fewer than 8 years at any duration are left out (147 of the 151 bundled stations). Missing years in the stations that remain are masked. The timing below is after one compile call, matching the published tutorial.


In [ ]:
quebec = load_eccc_quebec()
stations = [
    IDFData.from_dataframe(group, year="year", depth=True, name=sid)
    for sid, group in quebec.groupby("station_id")
    if group[DURATION_TAGS].notna().sum().min() >= 8
]
fit_idf(stations, "general")  # compile
t0 = time.perf_counter()
all_fit = fit_idf(stations, "general")  # one batched optimisation, missing values masked
t_all = time.perf_counter() - t0
q100 = all_fit.quantile([1.0], [100])[:, 0, 0]
print(
    f"{len(stations)} stations with ≥ 8 years for every duration were fitted in {t_all:.2f} s "
    f"(after compilation); {all_fit.converged.mean():.0%} converged. The 100-year 1-h intensity "
    f"ranges from {q100.min():.0f} to {q100.max():.0f} mm/h (median {np.median(q100):.0f})."
)


In [ ]:
import subprocess
import sys

# Basemap tiles are optional. The fits above do not need contextily or pyproj.
map_ok = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "contextily", "pyproj"]) == 0

meta = quebec.drop_duplicates("station_id").set_index("station_id").loc[[s.name for s in stations]]
fig, ax = plt.subplots(figsize=(7.2, 7.0))
if map_ok:
    import contextily as cx
    from pyproj import Transformer

    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    mx, my = transformer.transform(meta.lon.to_numpy(), meta.lat.to_numpy())
    xy = (mx, my)
else:
    xy = (meta.lon.to_numpy(), meta.lat.to_numpy())
sc = ax.scatter(
    *xy,
    c=q100,
    cmap="viridis",
    s=32,
    edgecolors="white",
    linewidths=0.35,
    vmin=np.quantile(q100, 0.05),
    vmax=np.quantile(q100, 0.95),
    zorder=3,
)
cb = fig.colorbar(sc, ax=ax, shrink=0.65, pad=0.02)
cb.set_label("100-year 1-h intensity (mm/h)")
if map_ok:
    pad = 0.10
    dx, dy = xy[0].max() - xy[0].min(), xy[1].max() - xy[1].min()
    ax.set_xlim(xy[0].min() - pad * dx, xy[0].max() + pad * dx)
    ax.set_ylim(xy[1].min() - pad * dy, xy[1].max() + pad * dy)
    # CartoDB Positron requires an API key. Esri World Light Gray needs none.
    try:
        cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas, crs="EPSG:3857", attribution=False)
    except Exception as exc:
        print(f"Basemap tiles were not added ({exc}).")
    ax.set_axis_off()
    ax.text(
        0.0,
        -0.02,
        "Tiles © Esri — Esri, DeLorme, NAVTEQ",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=7,
        color="0.35",
        clip_on=False,
    )
else:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
ax.set_title("d-GEV 100-year 1-h rainfall, Québec ECCC stations")
fig.tight_layout()
plt.show()


## Caveats

- ECCC annual maxima mix station histories (composite records); check metadata before inference.
- The d-GEV assumes a common shape ξ across durations; the scaling test above is the check.
- Wald intervals for 100-year levels can under-cover; prefer profile or bootstrap intervals.
